### 1. **What is a Factory**

A **Factory** is:
- *A piece of code whose job is to create objects, so the rest of the system **does not care how those objects are created***

In simple terms:
- You ask for an object
- The factory decides which concrete class to instantiate
- Your code works with a common interface

You **separate object creation from object usage**

### 2. **Why object creation is a problem in real systems**
Object creation is not trivial when:
- Different classes need **different dependencies**
- Creation logic depends on
    - config
    - enviroment
    - user input
    - runtime conditions
- You want **pluggability (swap implementations)**
- You want **test doubles / mocks**

Factories centralize this complexity

### 3. Mental model "new" is the enemy (sometimes)

Direct instantiation:
- ``model = GPTModel(...)``

Problems:
- Hard-coded dependcy
- Impossible to swap withougt editing code
- Tightly coupled

**Factory approach**
``model = ModelFactory.create("gpt")``

Now:
- Code depends on abstraction, not implementation
- Create logic lives in one place
- Adding a new model does not change callers

### 4. **Factory Pattern Family (big picture)**

There is not one factory patter, but a family:
- **Simple Factory** - Centralize object creation
- **Factory method** - Let subclasses decice what to create
- **Abstract Factory** - Create familis of related objects
- **Registry Factory** - Plugin-style dynamic creation
- **Functional Factory** - Pythonic lightweight factories
- **Dependency-Injection Factory** - Production-grade object wirint

We'll go one by one, from simplest -> most powerful.

5. **Simple Factory (the starting point)**

**What it is:**

*A function (or class) that **returns object based on input***

**Example**:

In [4]:
class PaypalPayment:
    def pay(self):
        print("Pay via PayPal")
        
class StripePayment:
    def pay(self):
        print("Pay via Stripe")
        
def payment_factory(payment_type: str):
    if payment_type == "paypal":
        return PaypalPayment()
    elif payment_type == "stripe":
        return StripePayment()
    else:
        raise ValueError("Unknown payment type")
    
payment = payment_factory("paypal")
payment.pay()

Pay via PayPal


**What improved**
- Creation logic moved out
- Callers don't know about concrete classes

**What still bad**
- if/elif still grows
- Factory must be modified for every new type

### 6. **Dictionary-based Simple Factory**

Replace if/elif with a mapping:

In [5]:
_PAYMENT_REGISTRY = {
    "paypal": PaypalPayment,
    "stripe": StripePayment
}

def payment_factory(payment_type: str):
    try:
        return _PAYMENT_REGISTRY[payment_type]()
    except KeyError:
        raise ValueError("Unknown payment type")

**Why this is better**
- O(1) lookup
- Easy to extend
- Cleaner code

**Still a Simple Factory**
- One function
- Centralized logic

### 7. **Factory Method Pattern (classic OOP)**

**Problem it solves**
- *I want subclasses to decide what object gets created*

Instead of a factory function, **creation becomes a method taht suclasses override**


**Structure**
- Base class defines **factory method**
- Subclasses override it
- Client code calls base interface

In [10]:
from abc import ABC, abstractmethod

class Payment(ABC):
    @abstractmethod
    def pay(self): ...
    
class PaypalPayment(Payment):
    def pay(self):
        print("Paypal")
        
class StripePayment(Payment):
    def pay(self):
        print("Stripe")
        
class PaymentProcessor(ABC):
    @abstractmethod
    def create_payment(self) -> Payment: ...
    
    def process(self):
        payment = self.create_payment()
        payment.pay()
        
        
class PaypalProcessor(PaymentProcessor):
    def create_payment(self):
        return PaypalPayment()
    
class StripeProcessor(PaymentProcessor):
    def create_payment(self) -> Payment:
        return StripePayment()
        
        
processor = PaypalProcessor()
processor.process()

Paypal


**Key insight**

The **factory is now a method**, not a function

**Why this exists**
- Frameworks use thsi a lot
- Template method patter + factory method
- Contro flos stays in base class

**When to use Factory Method**
- You have a **framework-like base class**
- Subclasses specialize object creation
- You want to enforce a workflow

### 8. **Abstract Factory Patter (factores that create families)**

**Problem it solves**
- *I need to create **multiple related objects** that must be compatible*

Example:
- UI toolkit
    - Button
    - Checkbox
    - TextField

- Each OS needs its own family


**Structure**
- One factory creates **multiple realted objects**
- Concrete factories produce consistent families

In [14]:
from abc import ABC, abstractmethod

# Abstract products
class Button(ABC):
    @abstractmethod
    def render(self): ...
    
class Checkbox(ABC):
    @abstractmethod
    def render(self): ...
    
    
# Concrete products
class WindowsButton(Button):
    def render(self):
        print("Windows Button")
        
class WindowsCheckbox(Checkbox):
    def render(self):
        print("Windows Checkbox")
        
class MacButton(Button):
    def render(self):
        print("Mac button")
        
class MacCheckbox(Checkbox):
    def render(self):
        print("Mac Checkbox")
        
# Abstract factory
class UIFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...
    
    @abstractmethod
    def create_checkbox(self) -> Checkbox: ...
    

# Concrete factories
class WindowsFactory(UIFactory):
    def create_button(self) -> Button:
        return WindowsButton()
    
    def create_checkbox(self) -> Checkbox:
        return WindowsCheckbox()
    
class MacFactory(UIFactory):
    def create_button(self) -> Button:
        return MacButton()
    
    def create_checkbox(self) -> Checkbox:
        return MacCheckbox()
    
    
factory = WindowsFactory()
button = factory.create_button()
checkbox = factory.create_checkbox()

**When Abstract Factory is justified**
- Objects must be **used togehter**
- You support **multiple platforms / enviroments**
- Compatibility matters

**Avoid it when**:
- You oncly create one object
- Complexity outweight benefits

### 9. **Registry-based Factory (PLUGIN ARCHITECTURE)**
This is extremely common in ML systems, agents, pipelines

**Idea**
- Objects register themselves
- Facotory doesn't know concrete classes
- New implementations can be added withogut modifie factory code

In [16]:
class Model:
    def run(self): ...
    
_MODEL_REGISTRY = {}

def register_model(name: str):
    def decorator(cls):
        _MODEL_REGISTRY[name] = cls
        return cls
    return decorator

@register_model("gpt")
def GPTModel(Model):
    def run(self):
        print("Running GPT")
        
@register_model("claude")
class ClaudeModel(Model):
    def run(self):
        print("Running Claude")
        
def model_factory(name: str) -> Model:
    try:
        return _MODEL_REGISTRY[name]()
    except KeyError:
        raise ValueError(f"Uknown model {name}")

**Why this is powerful**

- Open/Closed principle
- Plugin archirecture
- Perfect for:
    - ML models
    - OCR pipelines
    - Data loaders
    - Parsers

### 11. **Dependency-Injection Factory (production-grade)**

**Idea**
- Factory build objects
- Injects dependencies
- Manages lifecycle

In [17]:
class DB:
    ...
    
class Cache:
    ...
    
class Service:
    def __init__(self, db: DB, cache: Cache):
        self.db = db
        self.cache = cache
        
class ServiceFactory:
    def __init__(self, db: DB, cache: Cache):
        self.db = db
        self.cache = cache
        
    def create(self) -> Service:
        return Service(self.db, self.cache)

### **When not to use factories**

- Object creation is trivial
- Only one implementation exists
- readability suffers

Factories add inderection - use only when needed

### Decition table

- 2-3 options, simple - **Simple Factory**
- Framework / workflow = **Factory Method**
- Platform families - **Abstract Factory**
- Plugins / ML models - **Registry Factory**
- FastAPI services- **DI Factory**
- small scripts - **No factory**